In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers datasets torch onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 100.9 MB/s eta 0:00:00


In [10]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


In [17]:
!pip install -q onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.9 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import json, os

os.makedirs('/root/.kaggle', exist_ok=True)

# Paste your actual kaggle.json content below
kaggle_json = """{"username": "oxshubhamsingh", "key": "KGAT_6a9f175d899478610ee63217677c7630"}"""

Path('/root/.kaggle/kaggle.json').write_text(kaggle_json)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle API configured successfully!")

Kaggle API configured successfully!


In [5]:
!kaggle datasets download -d saurabhshahane/fake-news-classification -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification
License(s): Attribution 4.0 International (CC BY 4.0)
100% 92.1M/92.1M [00:05<00:00, 16.2MB/s]



In [6]:
import json, os, pandas as pd
from pathlib import Path

def preprocess_welfake_for_extension(csv_path):
    print("Reading dataset... this might take a minute.")
    df = pd.read_csv(csv_path)

    # 1. Clean missing values
    df['title'] = df['title'].fillna('')
    df['text'] = df['text'].fillna('')

    # 2. Extract Title + short body snippet (keeps ONNX inference lightweight)
    df['combined_input'] = df['title'] + " " + df['text'].str.slice(0, 300)

    # 3. Label Alignment: Original WELFake is 0=Fake, 1=Real.
    # Your script expects 1=Fake, 0=Real. Let's flip them to stay compatible:
    df['label_mapped'] = df['label'].apply(lambda x: 1 if x == 0 else 0)

    # 4. Shuffle and split (85% train, 15% validation)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    split_idx = int(len(df) * 0.85)

    train_df = df.iloc[:split_idx]
    val_df = df.iloc[split_idx:]

    # 5. Export to JSONL format
    for name, target_df in [('train', train_df), ('val', val_df)]:
        out_path = Path(f'/content/data/{name}.jsonl')
        out_path.write_text('\n'.join(
            target_df.apply(lambda r: json.dumps({"text": r['combined_input'], "label": r['label_mapped']}), axis=1)
        ))
        print(f"✅ Successfully saved {len(target_df)} records to {out_path}")

# Run the preprocessing
preprocess_welfake_for_extension('/content/data/WELFake_Dataset.csv')

Reading dataset... this might take a minute.
✅ Successfully saved 61313 records to /content/data/train.jsonl
✅ Successfully saved 10821 records to /content/data/val.jsonl


In [14]:
from pathlib import Path

train_script_content = """
import argparse
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="bert-base-uncased")
    parser.add_argument("--train_file", type=str, required=True)
    parser.add_argument("--validation_file", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    return parser.parse_args()

def main():
    args = parse_args()

    # 1. Load Custom JSONL Dataset
    dataset = load_dataset("json", data_files={
        "train": args.train_file,
        "validation": args.validation_file
    })

    # 2. Tokenization Setup
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 3. Model Definition (Binary Classification: Fake vs Real)
    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=2)

    # 4. Metrics Definition
    metric = evaluate.load("accuracy")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    # 5. Optimized Training Configuration for Colab T4 GPU
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none"
    )

    # 6. Initialize Trainer (Fixed: tokenizer -> processing_class)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("🚀 Starting model training on device:", training_args.device)
    trainer.train()

    print(f"💾 Saving fine-tuned model weights to {args.output_dir}")
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print("✅ Training successfully complete!")

if __name__ == "__main__":
    main()
"""

# Overwrite the script in your Drive
target_dir = Path('/content/drive/MyDrive/VeriSight_AI')
target_dir.mkdir(parents=True, exist_ok=True)
(target_dir / 'train.py').write_text(train_script_content.strip())
print("✅ Fixed train.py with processing_class standard!")

✅ Fixed train.py with processing_class standard!


In [18]:
from pathlib import Path

export_script_content = """
import argparse
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_dir", type=str, required=True, help="Directory containing the fine-tuned model")
    parser.add_argument("--onnx_path", type=str, required=True, help="Target file path to output the .onnx file")
    return parser.parse_args()

def main():
    args = parse_args()

    print(f"📦 Loading fine-tuned weights from {args.model_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(args.model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(args.model_dir)
    model.eval()

    # Generate dummy input tensors
    dummy_text = "This is a dummy news headline marker payload string."
    inputs = tokenizer(dummy_text, return_tensors="pt", truncation=True, max_length=256)

    input_names = ["input_ids", "attention_mask"]
    output_names = ["logits"]

    dynamic_axes = {
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    }

    os.makedirs(os.path.dirname(args.onnx_path), exist_ok=True)

    print(f"⚙️ Compiling model structure to ONNX format at {args.onnx_path}...")
    with torch.no_grad():
        torch.onnx.export(
            model,
            args=(inputs["input_ids"], inputs["attention_mask"]),
            f=args.onnx_path,
            export_params=True,
            input_names=input_names,
            output_names=output_names,
            dynamic_axes=dynamic_axes,
            opset_version=14,
            dynamo=False # Fixed: Forces standard robust conversion tracking
        )
    print("✅ Model exported successfully to ONNX graph format!")

if __name__ == "__main__":
    main()
"""

# Overwrite the script in your Drive
(Path('/content/drive/MyDrive/VeriSight_AI') / 'export_onnx.py').write_text(export_script_content.strip())
print("✅ Updated export_onnx.py with export safety patches!")

✅ Updated export_onnx.py with export safety patches!


In [15]:
!python /content/drive/MyDrive/VeriSight_AI/train.py \
    --model_name bert-base-uncased \
    --train_file /content/data/train.jsonl \
    --validation_file /content/data/val.jsonl \
    --output_dir /content/drive/MyDrive/VeriSight_AI/model_output

Loading weights: 100% 199/199 [00:00<00:00, 904.14it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing

In [19]:
!python /content/drive/MyDrive/VeriSight_AI/export_onnx.py \
    --model_dir /content/drive/MyDrive/VeriSight_AI/model_output \
    --onnx_path /content/drive/MyDrive/VeriSight_AI/model_output/model.onnx

📦 Loading fine-tuned weights from /content/drive/MyDrive/VeriSight_AI/model_output...
Loading weights: 100% 201/201 [00:00<00:00, 1177.90it/s, Materializing param=classifier.weight]
⚙️ Compiling model structure to ONNX format at /content/drive/MyDrive/VeriSight_AI/model_output/model.onnx...
/content/drive/MyDrive/VeriSight_AI/export_onnx.py:37: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:171: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a 